<a href="https://colab.research.google.com/github/vanderbilt-data-science/MNPSCollaborative/blob/Restart-From-Hackathon_v2.0/mnps_post_getting_started%20v4.3.0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **MNPS Job Equity Post Mini-Hackathon 4.3.0**
> A notebook to help you get started  
> DSI DSSG + MNPS Hackathon  
> September 15, 2025  
> Drafted by Wayne Birch - [contact him](wayne.birch@mnps.org) for questions, code update needs, or other questions about the notebook!

This notebook is a restart point based on the work done in the mini Hackathon with Metro Nashville Public Schools (MNPS) and the VU Data Science Institute (VU DSI).

 **Competition Details from the Hackathon with some updates follow:**

You aren't constrained to what is in this notebook, and please feel free to use your creativity to deliver the best solution
# **1** **| Addressed in 4.3.0**
* **Accountant vs Supervisor confusion:** We add supervisory verb detection and scope signals; if the JD emphasizes coaching/staff oversight and finance terms are light, the guardrail flips away from Accountant. If both appear, supervision strength + eligibility dominates; otherwise finance density dominates.
* **When to use Specialist vs Analyst** TAnalyst is now triggered by analysis/modeling/forecast/reporting signals; Specialist by hands-on operational verbs. A hard minimum-bounds penalty prevents Specialist from being the “easy default” when the job doesn’t meet Specialist minima.
* **Director not selected** If strategy/policy/portfolio/multi-team/district-wide cues are present and Director minima are met, the shortlist ordering plus the post-validator promotes Director. (If Director minima aren’t met, it will stick to Manager/Supervisor.).
* **Minimum bounds enforcement** TBefore the LLM sees candidates, we filter the shortlist to roles whose education + years + required license/cert are satisfied. After the LLM responds, we validate again, and if the pick fails minima, we re-select from eligible roles. This directly addresses Specialist overuse when prerequisites aren’t met.

## **2** | Environment Setup
Again, you're completely free to just download this notebook, create a local virtual environment and get to coding in your favorite IDE. We provide this code just as a rapid method to get started, and focus our efforts on implementation through Google Colab.

### **2a** | API Key Setup
#### **2a.1** | Access
The DSI has provided you an API key which can access **some** of the OpenAI models. These include:
* All versions of gpt-4o
* All versions of gpt-4.1
* All versions of o3-mini

Vector store upload, web search, code interpreter, and other functionality outside of the Chat Completions and Messages API is **not** supported. If you really want to use these things, you will have to make a good and cost-supported argument. If you don't feel like arguing, you can also utilize your own OpenAI API key.

#### **2a.2** | API Keys in Google Colab
To use your API key, click on the key icon (looks sort of like 🔑) in the left sidebar.  Under **Name**, add `OPENAI_API_KEY`. Under **Value**, paste your API key. Your API key is a jumble of numbers and letters, maybe even other symbols. Click the slider checkbox to enable **Notebook access** (so your notebook will grab these values without asking you).  

### **2b** | Runtime setup
We're going to install some packages in your environment so that you have access to the code functionality. If you need more packages, install more packages. Install **only** packages you trust.

In [ ]:
#Cell 3
!pip install -U openai

In [ ]:
#Cell 3.5
# ===== Environment Setup (single source of truth) =====
import os
from typing import List
import pandas as pd
from pydantic import BaseModel, Field
from google.colab import userdata

# 1) API key from Colab's 🔑 panel
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# 2) Read the model selector from Colab's 🔑 panel (can be alias or snapshot)
RAW_MODEL = userdata.get("OPENAI_MODEL")  # e.g., gpt-4o, gpt-4o-2024-11-20, gpt4.1, o3 mini

def normalize_model_id(s: str | None) -> str | None:
    if not s:
        return None
    s = s.strip().lower().replace("_", "-").replace(" ", "-")
    fixes = {
        "gpt4o": "gpt-4o",
        "gpt-4o": "gpt-4o",
        "gpt4.1": "gpt-4.1",
        "gpt-41": "gpt-4.1",
        "o3mini": "o3-mini",
        "o3-mini": "o3-mini",
    }
    return fixes.get(s, s)

alias_or_snapshot = normalize_model_id(RAW_MODEL)

# 3) Map aliases → pinned snapshots you prefer (edit to taste)
SNAPSHOTS = {
    # GPT-4o snapshots (stable; good for Structured Outputs)
    "gpt-4o":  "gpt-4o-2024-11-20",
    # GPT-4.1 family snapshot (long context)
    "gpt-4.1": "gpt-4.1-2025-04-14",
    # Keep o3-mini as an alias (no public dated snapshot ID); good for reasoning
    "o3-mini": "o3-mini",
}

# 4) Final MODEL_ID selection rule:
#    - If user entered an alias, pin it via SNAPSHOTS
#    - If user entered a snapshot, pass it through
#    - Else fallback to a safe default snapshot
MODEL_ID = SNAPSHOTS.get(alias_or_snapshot or "", None) or (alias_or_snapshot) or "gpt-4o-2024-11-20"

print("🔧 OPENAI_MODEL (raw):", RAW_MODEL)
print("✅ Using MODEL_ID:", MODEL_ID)


In [ ]:
# ===== Cell 4 — Unique run folder + get inputs (3 files) + robust CSV read + upload to OpenAI =====
import os, json, shutil, datetime as dt, zipfile
from pathlib import Path
import pandas as pd
from google.colab import drive
from openai import OpenAI

# ---------- 0) Mount Drive ----------
drive.mount('/content/drive')

# ---------- 1) Fixed output location (as requested) ----------
RUN_ROOT = Path("/content/drive/My Drive/Colab Notebooks/Run Results")
timestamp = dt.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
RUN_DIR = RUN_ROOT / f"RUN_{timestamp}"
INPUTS_DIR = RUN_DIR / "inputs"
OUTPUTS_DIR = RUN_DIR / "outputs"
for p in (RUN_DIR, INPUTS_DIR, OUTPUTS_DIR):
    p.mkdir(parents=True, exist_ok=True)

print("🗂️ Run folder:", RUN_DIR)

# ---------- 2) Where to find your three inputs by default ----------
# If you want to upload instead of copying from Drive, set ALLOW_UPLOAD = True.
DATA_INPUTS_DIR = Path("/content/drive/My Drive/Colab Notebooks/Data Inputs")
ALLOW_UPLOAD = False  # set True to be prompted to upload the 3 files from your computer

REQUIRED = {
    "Ground Truth Masterfile.csv": DATA_INPUTS_DIR / "Ground Truth Masterfile.csv",
    "New Sample_08.07.2025.csv":  DATA_INPUTS_DIR / "New Sample_08.07.2025.csv",
    "MNPS_Prompt_Resources.zip":  DATA_INPUTS_DIR / "MNPS_Prompt_Resources.zip",
}

# (A) Optionally upload files instead of copying from Drive
if ALLOW_UPLOAD:
    from google.colab import files as colab_files
    print("🔼 Upload the three files when prompted:")
    uploaded = colab_files.upload()  # opens a browser picker
    for name in REQUIRED.keys():
        if name in uploaded:
            dst = INPUTS_DIR / name
            with open(dst, "wb") as f:
                f.write(uploaded[name])
            REQUIRED[name] = dst  # point to the just-uploaded copy

# (B) Copy from Drive if not already present in /inputs
missing = []
for name, src in REQUIRED.items():
    dst = INPUTS_DIR / name
    if dst.exists():
        continue
    if src.exists():
        shutil.copy2(src, dst)
        print(f"📄 Copied: {src}  →  {dst}")
    else:
        missing.append(name)

if missing:
    raise FileNotFoundError(
        "These input files were not found. Place them in "
        f"{DATA_INPUTS_DIR} or enable ALLOW_UPLOAD=True:\n - " + "\n - ".join(missing)
    )

# ---------- 3) Unpack the resources zip into inputs/resources (optional but helpful) ----------
resources_zip = INPUTS_DIR / "MNPS_Prompt_Resources.zip"
RESOURCES_DIR = INPUTS_DIR / "resources"
if resources_zip.exists():
    RESOURCES_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(resources_zip, "r") as zf:
        zf.extractall(RESOURCES_DIR)
    print("🧰 Unpacked resources to:", RESOURCES_DIR)

# ---------- 4) Robust CSV reader (handles cp1252/latin1) ----------
def read_csv_smart(path: Path, **kwargs) -> pd.DataFrame:
    trials = [
        dict(encoding="utf-8"),
        dict(encoding="utf-8-sig"),
        dict(encoding="cp1252"),
        dict(encoding="latin1"),
    ]
    for t in trials:
        try:
            df = pd.read_csv(path, **{**t, **kwargs})
            print(f"✅ Read {path.name} with encoding={t['encoding']}")
            return df
        except UnicodeDecodeError:
            continue
    # last resort
    df = pd.read_csv(path, encoding="latin1", on_bad_lines="skip", **kwargs)
    print(f"⚠️ Read {path.name} with encoding=latin1 (on_bad_lines='skip')")
    return df

# Smoke test: load one row from the sample CSV (row 0) and build job_desc_text for downstream cells
sample_csv = INPUTS_DIR / "New Sample_08.07.2025.csv"
df = read_csv_smart(sample_csv)

required_cols = [
    "Job Description Name","Position Summary","Education","Work Experience",
    "Essential Functions","Licenses and Certifications","Knowledge, Skills and Abilities"
]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in {sample_csv.name}: {missing_cols}")

ROW_IDX = 0
r = df.iloc[ROW_IDX]
job_desc_text = f"""Job Description Name: {r['Job Description Name']}

Position Summary: {r['Position Summary']}
Education: {r['Education']}
Work Experience: {r['Work Experience']}
Licenses and Certifications: {r['Licenses and Certifications']}
Essential Functions: {r['Essential Functions']}
Knowledge, Skills and Abilities: {r['Knowledge, Skills and Abilities']}
"""
print("🧪 Prepared job_desc_text from row", ROW_IDX)

# ---------- 5) Upload the two CSVs to OpenAI so later cells can attach them ----------
client = OpenAI()  # API key already set in your Environment Setup cell
to_upload = [
    INPUTS_DIR / "Ground Truth Masterfile.csv",
    INPUTS_DIR / "New Sample_08.07.2025.csv",
]
uploaded = []
for p in to_upload:
    with open(p, "rb") as f:
        up = client.files.create(file=f, purpose="assistants")
    uploaded.append(up)

file_ids = [u.id for u in uploaded]  # <-- used by the Responses API cell later
print("⬆️ Uploaded file_ids:", file_ids)

# ---------- 6) Write a small manifest so you can audit each run ----------
manifest = {
    "run_folder": str(RUN_DIR),
    "created_utc": timestamp,
    "inputs": [str(p) for p in (INPUTS_DIR / "Ground Truth Masterfile.csv",
                                 INPUTS_DIR / "New Sample_08.07.2025.csv")],
    "resources_dir": str(RESOURCES_DIR) if RESOURCES_DIR.exists() else None,
    "uploaded_file_ids": file_ids,
}
(RUN_DIR / "RUN_METADATA.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("\n📁 Current run tree (first few entries):")
for i, p in enumerate(sorted(RUN_DIR.rglob("*"))):
    print(" -", p.relative_to(RUN_DIR))
    if i > 25:
        print(" … (truncated)")
        break

In [ ]:
# Cell 6
from openai import OpenAI
client = OpenAI()

visible = {m.id for m in client.models.list().data}
if MODEL_ID not in visible:
    print(f"⚠️ {MODEL_ID} is not visible to your key. "
          "Use an alias you do see (e.g., gpt-4o) or confirm access in your org.")
else:
    print(f"👍 {MODEL_ID} is available.")


## **3** | The Data

The current prompt is a two-step prompt that is successful through the ChatGPT interface. It requires two types of data:
* The data to be classified
* Supporting resources

We need to read all of this in. Let's grab it and use it. The first thing you'll do is just straight up download a zip file of all of this information.

You can download all of the reference files from the link provided, then upload in the sidebar. You'll then unzip the directory using the code below.

Click on the folder icon in the left sidebar (kinda looks like this 🗂️) and you'll see all the files there. We'll read them in.


In [ ]:
# Cell 8
from google.colab import drive
drive.mount('/content/drive')
base_target_folder = '/content/drive/My Drive/Colab Notebooks/Data Inputs'
!unzip "{base_target_folder}/MNPS_Prompt_Resources.zip" -d /content/

## **4** | The Prompts

What we have here is a direct prompt to get the response that we're looking for. We'll make this happen directly using the OpenAI Chat Completions API. Note that you can use other APIs as you like.

In [ ]:
#===================== Cell 12 (Zero-Shot Prompt) =============================
zero_shot_prompt = r"""
Objective: Evaluate and group jobs from the input file based on similarities in job functions, not job titles.

Process:
- Compare jobs using these attributes: Education, Work Experience, Licenses/Certifications, Essential Functions, Knowledge, Skills, Abilities, and Position Summary.
- Compare each job with reference sources using the same attributes (MNPS Roles and MNPS KSACs).
- Group jobs into:
  • Major role groupings from the comprehensive MNPS Roles list (e.g., Specialist, Analyst, Manager, Technician, Para Pro, Advisor, Partner, Supervisor, Director, etc.)
  • Minor sub-groupings: Lead, I, II, III (no IV; interpret IV/4 as Lead only if leadership/mentorship/escalation signals are present; otherwise III).

CRITICAL HARD RULE (Minimum Bounds):
- A candidate role is only eligible if the job meets that role’s minimum requirements for:
  • Education, AND
  • Years of relevant experience, AND
  • Required licenses/certifications (if any).
- If the minima are not met, that role must NOT be selected.

Disambiguation Guidance:
- Supervisor/Manager/Director vs Accountant/Analyst/Specialist:
  • If the description emphasizes supervision/people leadership (assigns work, evaluates staff, mentors, schedules, hires), prefer Supervisor/Manager/Director (depending on scope), not Accountant/Analyst/Specialist.
  • If district/division-wide strategy/policy/portfolio scope is evident and minima are met, prefer Director.
  • Strong GL/AP/AR/close/audit/GAAP signals indicate Accountant; strong modeling/metrics/forecasting/dashboard signals indicate Analyst; hands-on process/configuration/troubleshooting/ticketing indicates Specialist.
- KSAC alignment over title keywords.

Output Format (STRICT JSON for each row; do not add extra fields):
- major_role_group
- minor_sub_group (Lead, I, II, III; no IV)
- new_job_title (format “[Function] [Role] [Level]”, e.g., “Collections Specialist II”)
- grouping_justification (cite MNPS Roles/KSACs explicitly, plus the specific job attributes that drove the decision)

Job Title Convention:
- “[Function] [Role] [Level]” (Role must come from the MNPS Roles list).

Additional Guidelines:
- Cite sources (e.g., “MNPS KSACs: Technician”, “MNPS Roles”).
- Focus on the nature of work performed, not original titles.
- Consider complexity, responsibility, and required competencies.
- Use qualitative, holistic assessment centered on KSAC alignment (not a rigid numeric scoring).
"""


In [ ]:
#Cell 14
from pydantic import BaseModel, Field

class JobClassification(BaseModel):
    """Represents the classification of a job based on its functions."""
    job_title_original: str = Field(..., description="The original job title as provided in the input data using the job title convention specified.")
    new_job_title: str = Field(..., description="The proposed new job title based on the classification using the job title convention specified.")
    major_role_group: str = Field(..., description="The major grouping of the job based on its functional role (e.g., Specialist, Analyst, Manager).")
    minor_sub_group: str = Field(..., description="The minor sub-grouping within the major role group (e.g., Specialist I, II, III, IV).")
    grouping_justification: str = Field(..., description="The justification for placing the job in the specific major and minor groups, referencing job attributes and relevant documents.")

Create classifications using OpenAI. Of note here is:
* The **developer** prompt - this is the "system prompt" or "custom instructions" for the model. This determines the overall behavior of the model.
* The **user** prompt - this is what we send to the model like when we're chatting with ChatGPT.

In [ ]:
# Cell 15 — Classification Schema + Eligibility Minima, Role Guardrails & Prompt Builder
from typing import List
import re, numpy as np, pandas as pd
from pathlib import Path
from pydantic import BaseModel, Field

# --- Your existing/compatible Pydantic models ---
class JobClassification(BaseModel):
  major_role_group: str
  minor_sub_group: str
  new_job_title: str
  grouping_justification: str

class JobClassificationTable(BaseModel):
  """The table classification and overall commentary on the groupings provided by the AI system."""
  job_classification_table: List[JobClassification] = Field(..., description="The table of job classifications.")
  narrative_rationale: str = Field(..., description="The narrative commentary on the groupings provided by the AI system.")

# -------- Eligibility minima & role-signal helpers (NEW) ----------
# Paths (use your earlier variables if present; fall back to /content)
def _p(*candidates):
    for p in candidates:
        if isinstance(p, Path) and p.exists():
            return p
    return candidates[0]

RESOURCES_DIR = globals().get("RESOURCES_DIR", Path("/content"))
INPUTS_DIR    = globals().get("INPUTS_DIR", Path("/content"))

ROLES_CSV = _p(RESOURCES_DIR / "MNPS Roles.csv", INPUTS_DIR / "MNPS Roles.csv")
KSACS_CSV = _p(RESOURCES_DIR / "MNPS KSACs.csv", INPUTS_DIR / "MNPS KSACs.csv")

def _read_csv_any(path: Path):
    # Use your read_csv_smart if available, else robust fallback
    if "read_csv_smart" in globals():
        try:
            return read_csv_smart(path)
        except Exception:
            pass
    for enc in ["utf-8","utf-8-sig","cp1252","latin1"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            continue
    return pd.read_csv(path, encoding="latin1", on_bad_lines="skip")

roles_df = _read_csv_any(ROLES_CSV) if ROLES_CSV.exists() else None
ksacs_df = _read_csv_any(KSACS_CSV) if KSACS_CSV.exists() else None

def _find_col(df, *need):
    cl = {c.lower(): c for c in df.columns}
    for k,v in cl.items():
        k2 = k.replace("_"," ")
        if all(term in k2 for term in [n.lower() for n in need]):
            return v
    return None

if roles_df is not None:
    role_name_col  = _find_col(roles_df, "role")
    edu_min_col    = _find_col(roles_df, "min", "education") or _find_col(roles_df, "education")
    exp_min_col    = _find_col(roles_df, "min", "experience") or _find_col(roles_df, "years", "experience")
    lic_req_col    = _find_col(roles_df, "license") or _find_col(roles_df, "cert")
    VALID_ROLES = set(roles_df[role_name_col].astype(str).str.strip()) if role_name_col else set()
else:
    role_name_col = edu_min_col = exp_min_col = lic_req_col = None
    VALID_ROLES = set()

EDU_ORDER = ["hs","high school","ged","aa","associate","ba","bs","bachelor","bachelors","baccalaureate",
             "ma","ms","master","masters","phd","doctor","doctorate","edd","jd","md"]
EDU_RANK = {k:i for i,k in enumerate(EDU_ORDER)}

def edu_level_rank(text):
    if not isinstance(text,str): return -1
    t = text.lower(); best = -1
    for k,r in EDU_RANK.items():
        if re.search(rf"\b{k}\b", t): best = max(best, r)
    return best

def extract_years_experience(text):
    if not isinstance(text,str): return np.nan
    t = text.lower()
    m_rng = re.search(r"(\d+(?:\.\d+)?)\s*[-–]\s*(\d+(?:\.\d+)?)\s*(years?|yrs?)", t)
    if m_rng:
        a,b = float(m_rng.group(1)), float(m_rng.group(2)); return (a+b)/2.0
    m = re.search(r"(\d+(?:\.\d+)?)\s*(\+\s*)?(years?|yrs?)", t)
    return float(m.group(1)) if m else np.nan

def licenses_present(job_text, required_text):
    if not isinstance(required_text,str) or not required_text.strip(): return True
    if not isinstance(job_text,str): return False
    reqs = re.split(r"[;,/\|]", required_text)
    t = job_text.lower()
    return all(req.strip().lower() in t for req in reqs if req.strip())

# Build minima table
if roles_df is not None and role_name_col:
    role_meta = roles_df[[c for c in [role_name_col, edu_min_col, exp_min_col, lic_req_col] if c in roles_df.columns]].copy()
    role_meta.columns = ["role_name"] + [c for c in role_meta.columns[1:]]
    role_meta["edu_min_rank"] = role_meta.get(edu_min_col, "").apply(edu_level_rank)
    role_meta["exp_min_years"] = pd.to_numeric(role_meta.get(exp_min_col, np.nan), errors="coerce")
    role_meta["lic_requirements"] = role_meta.get(lic_req_col, "")
    ROLE_MINIMA = {
        str(r.role_name).strip(): {
            "edu_rank": int(r.edu_min_rank) if pd.notna(r.edu_min_rank) else -1,
            "exp_years": float(r.exp_min_years) if pd.notna(r.exp_min_years) else np.nan,
            "lic": str(r.lic_requirements) if pd.notna(r.lic_requirements) else ""
        }
        for _, r in role_meta.iterrows()
    }
else:
    ROLE_MINIMA = {}

# Signal lexicons to disambiguate roles
SUPERVISION_SIGNALS = [
    "supervise","supervision","direct reports","hire","recruit","discipline","coach","mentor",
    "evaluate performance","appraisal","assign work","schedule staff","lead a team","manage staff","oversee team"
]
DIRECTOR_SCOPE_SIGNALS = [
    "strategy","strategic","policy","portfolio","district-wide","division","sets direction","long-term planning",
    "multi-team","multi site","budgets across","executive","governance"
]
FINANCE_ACCOUNTING_SIGNALS = [
    "general ledger","gl","reconcile","reconciliation","journal entries","ap","accounts payable","ar",
    "accounts receivable","trial balance","gaap","audit","closing","fixed assets","variance analysis","month-end","year-end","cash application"
]
ANALYST_SIGNALS = [
    "analyze","analysis","analytics","kpi","metrics","dashboard","sql","forecast","model","trend","insight","reporting","visualization"
]
SPECIALIST_SIGNALS = [
    "process","execute","coordinate","operate","configure","troubleshoot","ticket","workflow","procedure",
    "subject matter expert","sme","hands-on"
]

def count_hits(text, lex):
    t = (text or "").lower()
    return sum(1 for w in lex if w in t)

def normalize_minor(sub: str, fields: dict) -> str:
    s = "" if sub is None else str(sub).strip()
    def looks_like_lead():
        blob = " ".join([str(fields.get("Position Summary","")), str(fields.get("Essential Functions","")), str(fields.get("Knowledge, Skills and Abilities",""))]).lower()
        return any(x in blob for x in ["lead","team lead","mentors","mentorship","coaches","escalation","oversees","assigns work","guides staff","sme","principal","tier 3"])
    if s.lower() == "lead": return "Lead"
    if s in {"I","II","III"}: return s
    m = re.search(r"\b(I|II|III|IV|4)\b", s.upper())
    if m:
        v = m.group(1)
        if v in {"I","II","III"}: return v
        return "Lead" if looks_like_lead() else "III"
    return "Lead" if looks_like_lead() else "III"

def meets_minimums(fields: dict, role_name: str) -> bool:
    spec = ROLE_MINIMA.get(role_name, None)
    if not spec: return True
    edu_rank_job = max(
        edu_level_rank(fields.get("Education","")),
        edu_level_rank(fields.get("Position Summary","")),
        edu_level_rank(fields.get("Knowledge, Skills and Abilities",""))
    )
    exp_years_job = np.nanmax([
        extract_years_experience(fields.get("Work Experience","")),
        extract_years_experience(fields.get("Position Summary","")),
        extract_years_experience(fields.get("Essential Functions",""))
    ])
    lic_ok = licenses_present(
        " ".join([str(fields.get("Licenses/Certifications","")), str(fields.get("Position Summary","")), str(fields.get("Essential Functions",""))]),
        spec.get("lic","")
    )
    edu_ok = (edu_rank_job >= spec.get("edu_rank",-1)) if spec.get("edu_rank",-1) >= 0 else True
    exp_req = spec.get("exp_years", np.nan)
    exp_ok = (exp_years_job >= exp_req) if not np.isnan(exp_req) else True
    return bool(edu_ok and exp_ok and lic_ok)

def role_signal_weight(fields: dict, role_name: str) -> int:
    blob = " ".join([
        str(fields.get("Position Summary","")),
        str(fields.get("Essential Functions","")),
        str(fields.get("Knowledge, Skills and Abilities",""))
    ])
    sup = count_hits(blob, SUPERVISION_SIGNALS)
    dirsig = count_hits(blob, DIRECTOR_SCOPE_SIGNALS)
    fin = count_hits(blob, FINANCE_ACCOUNTING_SIGNALS)
    ana = count_hits(blob, ANALYST_SIGNALS)
    spec = count_hits(blob, SPECIALIST_SIGNALS)
    w = 0
    rl = (role_name or "").lower()
    if rl in {"supervisor","manager","director"}:
        w += 2*sup + 3*dirsig
        if fin > 4 and sup == 0: w -= 2
    if rl == "accountant": w += 3*fin - 2*sup
    if rl == "analyst":    w += 2*ana - 1*sup
    if rl == "specialist":
        w += 2*spec
        if not meets_minimums(fields, role_name): w -= 3
    if rl == "director":   w += 3*dirsig + 1*sup
    return w

def canonicalize_role(pred_role: str) -> str:
    if not pred_role: return ""
    pl = str(pred_role).strip().lower()
    for vr in VALID_ROLES:
        if pl == vr.lower(): return vr
    for vr in sorted(VALID_ROLES, key=len, reverse=True):
        if re.search(rf"\b{re.escape(vr.lower())}\b", pl): return vr
    toks = re.findall(r"[A-Za-z]+", pl)
    if toks:
        last = toks[-1]
        for vr in VALID_ROLES:
            if last == vr.lower(): return vr
    return pred_role

def post_validate_role(fields: dict, predicted_role: str) -> str:
    role = canonicalize_role(predicted_role)
    if not meets_minimums(fields, role):
        elig = [r for r in VALID_ROLES if meets_minimums(fields, r)]
        if elig:
            role = sorted(elig, key=lambda r: role_signal_weight(fields, r), reverse=True)[0]
        return role
    blob = " ".join([str(fields.get("Position Summary","")), str(fields.get("Essential Functions",""))]).lower()
    sup = count_hits(blob, SUPERVISION_SIGNALS)
    dirsig = count_hits(blob, DIRECTOR_SCOPE_SIGNALS)
    fin = count_hits(blob, FINANCE_ACCOUNTING_SIGNALS)
    ana = count_hits(blob, ANALYST_SIGNALS)
    if "accountant" in role.lower() and sup >= 2 and fin < 3:
        role = "Supervisor" if dirsig == 0 else "Manager"
    if dirsig >= 2 and ("Director" in VALID_ROLES) and meets_minimums(fields, "Director"):
        role = "Director"
    if "analyst" in role.lower() and ana < 2 and count_hits(blob, SPECIALIST_SIGNALS) >= 2 and "Specialist" in VALID_ROLES:
        role = "Specialist"
    if "specialist" in role.lower() and ana >= 3 and "Analyst" in VALID_ROLES and meets_minimums(fields, "Analyst"):
        role = "Analyst"
    return role

def traffic_light(fields: dict, role: str) -> str:
    eligible = meets_minimums(fields, role)
    blob = " ".join([
        str(fields.get("Position Summary","")), str(fields.get("Essential Functions","")),
        str(fields.get("Knowledge, Skills and Abilities","")), str(fields.get("Education","")),
        str(fields.get("Work Experience","")), str(fields.get("Licenses/Certifications",""))
    ])
    sup = count_hits(blob, SUPERVISION_SIGNALS)
    dirsig = count_hits(blob, DIRECTOR_SCOPE_SIGNALS)
    fin = count_hits(blob, FINANCE_ACCOUNTING_SIGNALS)
    ana = count_hits(blob, ANALYST_SIGNALS)
    spec = count_hits(blob, SPECIALIST_SIGNALS)
    score = 0
    if eligible: score += 2
    rl = (role or "").lower()
    if rl in {"supervisor","manager","director"}: score += sup + 2*dirsig
    if rl == "director": score += dirsig
    if rl == "accountant": score += 2*fin - sup
    if rl == "analyst": score += 2*ana - sup
    if rl == "specialist": score += 2*spec
    return "Green" if score >= 4 else ("Yellow" if score >= 2 else "Red")

# ---- Prompt text builder (uses Cell 12's zero_shot_prompt) ----
def full_text_for_row(r):
    def getv(col):
        try:
            v = r[col];
            return "" if pd.isna(v) else str(v)
        except Exception:
            return ""
    fields = {
        "Position Summary": getv("Position Summary"),
        "Essential Functions": getv("Essential Functions"),
        "Education": getv("Education"),
        "Work Experience": getv("Work Experience"),
        "Licenses/Certifications": getv("Licenses and Certifications"),
        "Knowledge, Skills and Abilities": getv("Knowledge, Skills and Abilities"),
    }

    if VALID_ROLES:
        eligible = [nm for nm in VALID_ROLES if meets_minimums(fields, nm)]
        shortlist = eligible if eligible else list(VALID_ROLES)
        shortlist = sorted(shortlist, key=lambda nm: role_signal_weight(fields, nm), reverse=True)[:20]
        candidate_block = "Candidate Roles (eligible): " + ", ".join(shortlist)
    else:
        candidate_block = "Candidate Roles (eligible): (role catalog not found — apply minima per policy)"

    job_blob = f"""Job Description Name: {getv('Job Description Name')}

Position Summary: {fields['Position Summary']}
Education: {fields['Education']}
Work Experience: {fields['Work Experience']}
Licenses and Certifications: {fields['Licenses/Certifications']}
Essential Functions: {fields['Essential Functions']}
Knowledge, Skills and Abilities: {fields['Knowledge, Skills and Abilities']}
"""

    policy = (
        "HARD RULE: Select ONLY from the eligible Candidate Roles. "
        "A role is eligible only if the job meets the role's minimum Education, Experience, and required Licenses/Certifications. "
        "Minor sub-groups allowed: Lead, I, II, III (map IV/4 to Lead if leadership signals are present; otherwise III)."
    )

    return (
        zero_shot_prompt.strip()
        + "\n\n" + policy
        + "\n\n" + candidate_block
        + "\n\nClassify the following job description:\n\n"
        + job_blob
    )


In [ ]:
# Cell 16 — Zero-Shot Classification (per row) + Post-Validation + Save CSV
import json, time, random
import pandas as pd

# --- Inputs/DataFrame setup ----------------------------------------------------
def read_csv_robust(path: str):
    for enc in ["utf-8","utf-8-sig","latin1","windows-1252"]:
        try: return pd.read_csv(path, encoding=enc)
        except Exception: continue
    return pd.read_csv(path, encoding="latin1", errors="ignore")

# Prefer an existing prepped DataFrame if one exists; else load from config
if "work_df" in globals() and isinstance(work_df, pd.DataFrame):
    in_df = work_df.copy()
else:
    if "BATCH_INPUT_CSV" not in globals():
        raise ValueError("BATCH_INPUT_CSV is not defined. Set it in Inputs & Configuration.")
    raw_df = read_csv_robust(BATCH_INPUT_CSV)
    required = [
        "Job Description Name","Position Summary","Education","Work Experience",
        "Essential Functions","Licenses and Certifications","Knowledge, Skills and Abilities"
    ]
    missing = [c for c in required if c not in raw_df.columns]
    if missing:
        raise ValueError(f"Missing required columns in Batch Input: {missing}")
    in_df = raw_df.reset_index().rename(columns={"index":"row_id"}).copy()
    work_df = in_df  # keep for later cells

# --- Output filenames ----------------------------------------------------------
OUTPUT_PRED_CSV = globals().get("OUTPUT_PRED_CSV", "classified_job_descriptions.csv")
DECISION_LOG_CSV = "classification_decision_log.csv"

# --- LLM call helper -----------------------------------------------------------
def _call_llm_safe(prompt: str) -> str:
    """
    Use user's existing call_llm_json if available; otherwise try a minimal fallback.
    Returns JSON string (or raises on failure).
    """
    if "call_llm_json" in globals():
        return call_llm_json(prompt)
    # Fallback: raise with guidance (most notebooks already define call_llm_json)
    raise RuntimeError("call_llm_json() is not defined. Please run the earlier cell that sets up the LLM client.")

# --- Per-row classification loop ----------------------------------------------
records = []
logs = []

# Temperature/model from earlier cells if present
TEMPERATURE = globals().get("TEMPERATURE", 0.2)
MODEL_ID    = globals().get("MODEL_ID", "gpt-4o-mini")

for _, row in in_df.iterrows():
    row_id = int(row.get("row_id", _))
    job_name = str(row.get("Job Description Name",""))

    try:
        # 1) Build prompt (uses zero_shot_prompt + eligibility shortlist; defined in Cell 15)
        prompt_text = full_text_for_row(row)

        # 2) Call LLM
        raw = _call_llm_safe(prompt_text)

        # 3) Parse JSON (tolerate markdown code fences)
        raw_str = str(raw).strip()
        if raw_str.startswith("```"):
            # strip code fences if present
            raw_str = raw_str.strip("`")
            # remove possible language hint like ```json
            first_newline = raw_str.find("\n")
            if first_newline != -1:
                raw_str = raw_str[first_newline+1:].strip()
        rec = json.loads(raw_str)

        # Expected keys: major_role_group, minor_sub_group, new_job_title, grouping_justification
        # Build fields for validators
        fields = {
            "Position Summary": row.get("Position Summary",""),
            "Essential Functions": row.get("Essential Functions",""),
            "Education": row.get("Education",""),
            "Work Experience": row.get("Work Experience",""),
            "Licenses/Certifications": row.get("Licenses and Certifications",""),
            "Knowledge, Skills and Abilities": row.get("Knowledge, Skills and Abilities",""),
        }

        # 4) Post-validate/canonicalize (from Cell 15)
        fixed_major = post_validate_role(fields, rec.get("major_role_group",""))
        fixed_minor = normalize_minor(rec.get("minor_sub_group",""), fields)

        # 5) Compose output row
        row_out = {
            "original_job_title": job_name,   # must mirror Job Description Name
            "new_job_title": rec.get("new_job_title",""),
            "major_role_group": fixed_major,
            "minor_sub_group": fixed_minor,
            "grouping_justification": rec.get("grouping_justification",""),
            "Job Description Name": job_name,
            "row_id": row_id
        }

        # Optional: confidence/traffic light
        try:
            row_out["status_color"] = traffic_light(fields, fixed_major)
        except Exception:
            row_out["status_color"] = ""

        records.append(row_out)

        logs.append({
            "row_id": row_id,
            "job_name": job_name,
            "prompt_preview": prompt_text[:1500],
            "raw_response": raw_str
        })

        # small jitter to be polite to API if running many rows
        time.sleep(0.05 + random.random()*0.05)

    except Exception as e:
        # Log and keep DF alignment
        err_msg = f"{type(e).__name__}: {e}"
        print(f"[Row {row_id} | {job_name}] Error -> {err_msg}")
        logs.append({
            "row_id": row_id,
            "job_name": job_name,
            "prompt_preview": (prompt_text[:1500] if 'prompt_text' in locals() else "N/A"),
            "raw_response": (raw if 'raw' in locals() else "N/A"),
            "error": err_msg
        })
        records.append({
            "original_job_title": job_name,
            "new_job_title": "Error",
            "major_role_group": "Error",
            "minor_sub_group": "Error",
            "grouping_justification": f"Error: {err_msg}",
            "Job Description Name": job_name,
            "row_id": row_id,
            "status_color": "Red"
        })

# --- Build DataFrame & save ----------------------------------------------------
out_df = pd.DataFrame(records)

# Ensure the canonical column order if present
cols_order = [
    "original_job_title","new_job_title","major_role_group","minor_sub_group",
    "grouping_justification","status_color","Job Description Name","row_id"
]
out_df = out_df[[c for c in cols_order if c in out_df.columns] + [c for c in out_df.columns if c not in cols_order]]

out_df.to_csv(OUTPUT_PRED_CSV, index=False, encoding="utf-8")
pd.DataFrame(logs).to_csv(DECISION_LOG_CSV, index=False, encoding="utf-8")

print(f"Saved predictions -> {OUTPUT_PRED_CSV}")
print(f"Saved decision logs -> {DECISION_LOG_CSV}")
print(f"Rows processed: {len(out_df)}")

# --- Build a parsed object for Cell 17 inspection -----------------------------
# Keep the same variable name 'parsed' so your Cell 17 can print it.
try:
    # Convert rows back into a JobClassificationTable for continuity
    parsed_rows = []
    for _, r in out_df.iterrows():
        try:
            parsed_rows.append(JobClassification(
                major_role_group = str(r.get("major_role_group","")),
                minor_sub_group  = str(r.get("minor_sub_group","")),
                new_job_title    = str(r.get("new_job_title","")),
                grouping_justification = str(r.get("grouping_justification",""))
            ))
        except Exception:
            # If any required field is missing, fill with safe defaults
            parsed_rows.append(JobClassification(
                major_role_group = str(r.get("major_role_group","Error")),
                minor_sub_group  = str(r.get("minor_sub_group","Error")),
                new_job_title    = str(r.get("new_job_title","Error")),
                grouping_justification = str(r.get("grouping_justification",""))
            ))
    narrative = "Batch classified using zero-shot prompt with eligibility minima and role guardrails; results saved to CSV."
    parsed = JobClassificationTable(job_classification_table=parsed_rows, narrative_rationale=narrative)
except Exception as e:
    print("Warning: could not build 'parsed' object for Cell 17:", e)

# Quick preview
display(out_df.head(10))


In [ ]:
# ===== Cell 16.45 — Pre-flight: are all prerequisites loaded for batch? =====
from pathlib import Path

print("Have MODEL_ID:", 'MODEL_ID' in globals(), (MODEL_ID if 'MODEL_ID' in globals() else None))
print("Have df:", 'df' in globals(), (len(df) if 'df' in globals() else None))
print("Have zero_shot_prompt:", 'zero_shot_prompt' in globals())
print("Have OUTPUTS_DIR:", 'OUTPUTS_DIR' in globals(), (OUTPUTS_DIR if 'OUTPUTS_DIR' in globals() else None))

if 'RUN_DIR' in globals():
    print("RUN_DIR:", RUN_DIR)
    print("Outputs path will be:", Path(OUTPUTS_DIR) / "Job_Classifications_Batch.csv")
else:
    print("RUN_DIR missing — re-run your unique run cell (Cell 4).")


In [ ]:
# ========= Cell 16.46 =====================
from pathlib import Path
(Path(OUTPUTS_DIR)/"Job_Classifications_Batch.csv").unlink(missing_ok=True)
(Path(OUTPUTS_DIR)/"Batch_Errors.json").unlink(missing_ok=True)


In [ ]:
# ===== Cell 16.5 — Batch v3.1 (DEBUG: loud logs, resume-safe, JSON fence fix) =====
from openai import OpenAI
from pathlib import Path
import pandas as pd, json, time, random, inspect, re, shutil

print("=== Batch v3.1 start ===")

# ---- prerequisites ----
assert 'df' in globals(), "Run Cell 4 first (loads df)."
assert 'OUTPUTS_DIR' in globals(), "Run the unique-run cell first."
assert 'MODEL_ID' in globals(), "Run Environment Setup first."
assert 'zero_shot_prompt' in globals(), "Define zero_shot_prompt (your prompt cell)."

print("MODEL_ID:", MODEL_ID)
print("Rows in df:", len(df))
print("OUTPUTS_DIR:", OUTPUTS_DIR)

# If available, show SDK version
try:
    import openai as _o
    print("openai SDK:", getattr(_o, "__version__", "(unknown)"))
except Exception:
    pass

client = OpenAI(timeout=60.0, max_retries=2)

# ---- robust CSV reader if you have it from Cell 4 ----
def _read_csv(path: Path, **kw):
    if 'read_csv_smart' in globals():
        return read_csv_smart(path, **kw)
    return pd.read_csv(path, encoding="utf-8", **kw)

# ---- tiny context from Ground Truth (once) ----
context_block = ""
gt_path = Path(INPUTS_DIR) / "Ground Truth Masterfile.csv" if 'INPUTS_DIR' in globals() else None
if gt_path and gt_path.exists():
    try:
        gt_df = _read_csv(gt_path).fillna("")
        preferred_cols = [
            "Original Job Title","New Job Title","Major Role Group","Minor Sub-Group","Justification for Grouping",
            "Position Summary","Education","Work Experience","Licenses and Certifications","Essential Functions","Knowledge, Skills and Abilities"
        ]
        cols = [c for c in preferred_cols if c in gt_df.columns] or list(gt_df.columns)[:8]
        mini = gt_df[cols].head(3)
        context_block = "Context (3 ground-truth examples):\n" + mini.to_json(orient="records", force_ascii=False)
        print("Context block chars:", len(context_block))
    except Exception as e:
        context_block = f"(Context unavailable: {e})"
        print("Context build error:", e)
else:
    print("No Ground Truth CSV found at", gt_path)

def build_job_text(r):
    def getv(col):
        try:
            v = r[col]
            return "" if pd.isna(v) else str(v)
        except Exception:
            return ""
    return f"""Job Description Name: {getv('Job Description Name')}

Position Summary: {getv('Position Summary')}
Education: {getv('Education')}
Work Experience: {getv('Work Experience')}
Licenses and Certifications: {getv('Licenses and Certifications')}
Essential Functions: {getv('Essential Functions')}
Knowledge, Skills and Abilities: {getv('Knowledge, Skills and Abilities')}
"""

def full_text_for_row(r):
    return (
        zero_shot_prompt.strip()
        + ("\n\n" + context_block if context_block else "")
        + "\n\nClassify the following job description:\n\n"
        + build_job_text(r)
    )

# ---- capability detection ----
def _has_param(obj, name: str) -> bool:
    try:
        return name in inspect.signature(obj).parameters
    except Exception:
        return False

supports_parse_schema  = _has_param(client.responses.parse,  "response_format")
supports_create_schema = _has_param(client.responses.create, "response_format")

print("supports_parse_schema:", supports_parse_schema,
      "| supports_create_schema:", supports_create_schema)

# ---- JSON sanitizers (strip ```json fences etc.) ----
_fence_re = re.compile(r"^\s*```(?:json)?\s*(.*?)\s*```\s*$", re.DOTALL|re.IGNORECASE)
_brace_re = re.compile(r"\{.*\}", re.DOTALL)

def coerce_to_json_str(raw: str) -> str:
    if not isinstance(raw, str):
        return ""
    s = raw.strip()
    m = _fence_re.match(s)
    if m:
        s = m.group(1).strip()
    if not s.startswith("{"):
        m2 = _brace_re.search(s)
        if m2:
            s = m2.group(0)
    return s

# ---- call wrapper ----
def call_model_with_text(text, temp, max_tokens):
    if supports_parse_schema:
        resp = client.responses.parse(
            model=MODEL_ID,
            input=[{"role": "user", "content": [{"type":"input_text","text": text}]}],
            temperature=temp,
            max_output_tokens=max_tokens,
            response_format=JobClassificationTable,
        )
        return resp.output_parsed, resp.output_text or ""
    elif supports_create_schema:
        schema = JobClassificationTable.model_json_schema()
        resp = client.responses.create(
            model=MODEL_ID,
            input=[{"role": "user", "content": [{"type":"input_text","text": text}]}],
            temperature=temp,
            max_output_tokens=max_tokens,
            response_format={
                "type":"json_schema",
                "json_schema":{"name":"JobClassificationTable","schema":schema,"strict":True},
            },
        )
        raw = getattr(resp, "output_text", None) or ""
        try:
            return JobClassificationTable.model_validate_json(raw), raw
        except Exception:
            cleaned = coerce_to_json_str(raw)
            return JobClassificationTable.model_validate_json(cleaned), cleaned
    else:
        schema_json = json.dumps(JobClassificationTable.model_json_schema(), indent=2)
        strict = (
            "You MUST return ONLY valid JSON that matches the following JSON Schema. No prose, no markdown.\n"
            f"JSON Schema:\n{schema_json}\n\nTask:\n{text}"
        )
        resp = client.responses.create(
            model=MODEL_ID,
            input=[{"role":"user","content":[{"type":"input_text","text": strict}]}],
            temperature=temp,
            max_output_tokens=max_tokens,
        )
        raw = getattr(resp, "output_text", None) or ""
        cleaned = coerce_to_json_str(raw)
        return JobClassificationTable.model_validate_json(cleaned), cleaned

def backoff_sleep(k): time.sleep(min(20, 1.8**k + random.random()))

# ---- batching parameters (start with a small limit to confirm) ----
ROW_START   = 0
ROW_LIMIT   = None          # ← first test; set to None after you see progress
TEMP        = 0.2
MAX_TOKENS  = 900
SAVE_EVERY  = 2
MAX_ATTEMPTS_PER_ROW = 3

# ---- resume: skip rows already saved ----
batch_csv_path = Path(OUTPUTS_DIR) / "Job_Classifications_Batch.csv"
processed = set()
if batch_csv_path.exists():
    try:
        prior = pd.read_csv(batch_csv_path, usecols=["source_row_index"])
        processed = set(prior["source_row_index"].astype(int).tolist())
        print(f"Resume mode: {len(processed)} rows already done; will skip them.")
    except Exception as e:
        print("Resume disabled (could not read prior batch CSV):", e)

# ---- plan iteration ----
end_idx = len(df) if ROW_LIMIT is None else min(len(df), ROW_START + ROW_LIMIT)
indexes = [i for i in range(ROW_START, end_idx) if i not in processed]
print(f"Planned rows to process: {len(indexes)} of {len(df)} (from {ROW_START} to {end_idx-1})")
if not indexes:
    print("Nothing to do: either ROW_LIMIT=0, or all planned rows already in batch CSV,")
    print("or ROW_START >= end_idx. If you want a clean re-run, delete previous batch files:")
    print(" (Path(OUTPUTS_DIR)/'Job_Classifications_Batch.csv').unlink(missing_ok=True)")
    print(" (Path(OUTPUTS_DIR)/'Batch_Errors.json').unlink(missing_ok=True)")

records, errors = [], []
start_time = time.time()

# ---- loop ----
for k, i in enumerate(indexes, start=1):
    r = df.iloc[i]
    text = full_text_for_row(r)

    t0 = time.time()
    parsed = None
    raw    = ""

    for attempt in range(MAX_ATTEMPTS_PER_ROW):
        try:
            parsed, raw = call_model_with_text(text, TEMP, MAX_TOKENS)
            break
        except Exception as e:
            msg = str(e)
            if attempt == MAX_ATTEMPTS_PER_ROW - 1:
                snippet = (coerce_to_json_str(raw) if raw else "")[:600]
                errors.append((i, "exception", msg[:500], snippet))
            backoff_sleep(attempt)

    if parsed:
        try:
            for rec in parsed.job_classification_table:
                row_out = rec.model_dump()
                row_out["source_row_index"] = i
                row_out["model_used"] = MODEL_ID
                records.append(row_out)
        except Exception as e:
            errors.append((i, "parse_collect_error", str(e)[:300], (raw or "")[:300]))
    else:
        cleaned = coerce_to_json_str(raw) if raw else ""
        errors.append((i, "no_parsed_output", cleaned[:600]))

    # checkpoint save
    if (k % SAVE_EVERY == 0) or (k == len(indexes)):
        if records:
            if batch_csv_path.exists():
                try:
                    prev = pd.read_csv(batch_csv_path)
                    merged = pd.concat([prev, pd.DataFrame(records)], ignore_index=True)
                    merged.drop_duplicates(subset=["source_row_index","job_title_original","new_job_title"], inplace=True)
                    merged.to_csv(batch_csv_path, index=False, encoding="utf-8")
                except Exception:
                    pd.DataFrame(records).to_csv(batch_csv_path, index=False, encoding="utf-8")
            else:
                pd.DataFrame(records).to_csv(batch_csv_path, index=False, encoding="utf-8")
            print(f"Checkpoint: wrote {len(pd.read_csv(batch_csv_path))} rows to batch CSV.")
            records = []
        Path(OUTPUTS_DIR, "Batch_Errors.json").write_text(json.dumps(errors, indent=2), encoding="utf-8")

    print(f"[{k}/{len(indexes)}] row {i} in {time.time()-t0:.1f}s | total {(time.time()-start_time)/60:.1f} min | "
          f"ok so far {k - len(errors)} | err {len(errors)}")

# copy batch → single so housekeeping/master sees it
if batch_csv_path.exists():
    dst = Path(OUTPUTS_DIR) / "Job_Classifications.csv"
    shutil.copy2(batch_csv_path, dst)
    print("📄 Copied batch to:", dst)

print("✅ Batch complete. Files in:", OUTPUTS_DIR)


In [ ]:
# ===== Cell 16.54 — Live peek while batch runs =====
from pathlib import Path
import pandas as pd

p = Path(OUTPUTS_DIR) / "Job_Classifications_Batch.csv"
if p.exists():
    dfb = pd.read_csv(p)
    print("Rows saved so far:", len(dfb))
    # show last few and a quick look at which source rows are pending
    display(dfb.tail(5))
    if "source_row_index" in dfb.columns and 'df' in globals():
        done = set(dfb["source_row_index"].astype(int))
        pending = [i for i in range(len(df)) if i not in done]
        print("Remaining rows:", len(pending), "| next up:", pending[:10])
else:
    print("No batch file yet at:", p)


In [ ]:
# ===== Cell 16.55 — Batch audit: counts, parameters, error preview =====
from pathlib import Path
import pandas as pd, json

assert 'OUTPUTS_DIR' in globals(), "Run Cell 4 first (creates OUTPUTS_DIR)."
assert 'df' in globals(), "Run Cell 4 first (loads df)."

print("Total rows in input df:", len(df))

batch_csv_path = Path(OUTPUTS_DIR) / "Job_Classifications_Batch.csv"
if batch_csv_path.exists():
    dfb = pd.read_csv(batch_csv_path)
    print("Rows saved in batch CSV:", len(dfb))
    if "source_row_index" in dfb.columns:
        done = sorted(dfb["source_row_index"].astype(int).unique().tolist())
        print("First 10 processed row indexes:", done[:10])
        print("Last 10 processed row indexes:", done[-10:])
    else:
        print("Note: 'source_row_index' column missing in batch CSV.")
else:
    print("⚠️ No batch CSV found at:", batch_csv_path)

errors_path = Path(OUTPUTS_DIR) / "Batch_Errors.json"
if errors_path.exists():
    try:
        errs = json.loads(errors_path.read_text())
        print("Error entries:", len(errs))
        for j, e in enumerate(errs[:5]):
            print(f"  {j+1}.", e if isinstance(e, str) else (e[0:2] if isinstance(e, list) else e))
    except Exception as e:
        print("Could not read Batch_Errors.json:", e)
else:
    print("No Batch_Errors.json present — either none failed or nothing ran.")


In [ ]:
# ===== Cell 16.6 — Quick sanity check for current run =====
from pathlib import Path
import pandas as pd, json

assert 'OUTPUTS_DIR' in globals(), "Run your unique-run cell first (defines OUTPUTS_DIR)."

batch = Path(OUTPUTS_DIR) / "Job_Classifications_Batch.csv"
if batch.exists():
    dfb = pd.read_csv(batch)
    print("✅ Batch rows in this run:", len(dfb))
    display(dfb.head(5))
else:
    print("⚠️ No batch file found at", batch)

errs = Path(OUTPUTS_DIR) / "Batch_Errors.json"
if errs.exists():
    e = json.loads(Path(errs).read_text())
    print("⚠️ Rows with errors:", len(e))
    if e:
        print("First error:", e[0])


In [ ]:
#Cell 17
# Inspect parsed output (Responses API)
try:
    parsed  # from Cell 16
    print(parsed.model_dump_json(indent=2))
except NameError:
    print("No 'parsed' object found. Run Cell 16 first.")


In [ ]:
# Cell 17.5 — Build a response_dict from the Responses API parsed object
from pathlib import Path
import json
import pandas as pd

# Make sure Cell 16 ran (it defines `parsed`) and the run folders exist
assert 'parsed' in globals(), "Run Cell 16 first (it sets `parsed`)."
assert 'OUTPUTS_DIR' in globals(), "Run the unique-run cell first (defines OUTPUTS_DIR)."

# Convert the Pydantic objects to plain dicts
response_dict = {
    "job_classification_table": [rec.model_dump() for rec in parsed.job_classification_table],
    "narrative_rationale": parsed.narrative_rationale,
}

# Optional: preview the first rows
display(pd.DataFrame(response_dict["job_classification_table"]).head(10))

# Optional: save a pretty JSON alongside your other outputs
out_json = Path(OUTPUTS_DIR) / "Parsed_Response.json"
out_json.write_text(json.dumps(response_dict, indent=2), encoding="utf-8")
print("Saved:", out_json)

# Also return the dict so it shows below the cell
response_dict


In [ ]:
#Cell 18
# Preview the saved classifications CSV (if present)
from pathlib import Path
import pandas as pd

csv_path = Path(OUTPUTS_DIR) / "Job_Classifications.csv"
if csv_path.exists():
    display(pd.read_csv(csv_path).head(10))
else:
    print("No Job_Classifications.csv found in", OUTPUTS_DIR)


In [ ]:
# Cell 19 ===== Housekeeping & Archive (Run Results) =====
# Place this cell at the END of the notebook. Run after your pipeline finishes.
from google.colab import drive
from pathlib import Path
import shutil, json, re
import datetime as dt
import pandas as pd

# ---------- CONFIG (edit to taste) ----------
RUN_ROOT = Path("/content/drive/My Drive/Colab Notebooks/Run Results")
ARCHIVE_DIR = RUN_ROOT / "_archives"
MASTER_DIR  = RUN_ROOT / "_master"

KEEP_LAST_N_RUNS   = 10     # keep this many newest runs; older ones can be deleted
ZIP_OLDER_RUNS     = True   # zip runs (into _archives) to save space
PURGE_RAW_JSON     = True   # delete outputs/Raw_Response.json inside each run
PURGE_PARSED_JSON  = False  # delete outputs/Parsed_Response.json
PURGE_BATCH_ERRORS = False  # delete outputs/Batch_Errors.json
SKIP_CURRENT_RUN   = True   # don't zip/purge/delete the most recent run
DRY_RUN            = True   # <<< safety: set False to actually apply changes

# ---------- Mount Drive (no-op if already mounted) ----------
drive.mount('/content/drive')

# ---------- Helpers ----------
def parse_run_ts(name: str):
    m = re.match(r"RUN_(\d{8}_\d{6})$", name)
    if not m:
        return None
    try:
        return dt.datetime.strptime(m.group(1), "%Y%m%d_%H%M%S")
    except Exception:
        return None

def folder_size_bytes(p: Path) -> int:
    total = 0
    for f in p.rglob("*"):
        if f.is_file():
            try:
                total += f.stat().st_size
            except Exception:
                pass
    return total

def human_mb(nbytes: int) -> str:
    return f"{nbytes/1_000_000:.2f} MB"

# ---------- Discover run folders ----------
runs = []
for d in RUN_ROOT.iterdir():
    if d.is_dir() and d.name.startswith("RUN_"):
        ts = parse_run_ts(d.name)
        if ts:
            runs.append((d, ts))

runs.sort(key=lambda x: x[1], reverse=True)  # newest first
print(f"Found {len(runs)} run folders under: {RUN_ROOT}")

current = runs[0][0] if runs else None
if current:
    print("Most recent run:", current.name)

# Summary of the first few
for d, ts in runs[:5]:
    print(f" - {d.name} | {ts:%Y-%m-%d %H:%M:%S} | size≈ {human_mb(folder_size_bytes(d))}")

# Ensure archive/master dirs
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
MASTER_DIR.mkdir(parents=True, exist_ok=True)

# ---------- Plan actions ----------
actions = []

# 1) Purge large intermediates within runs
def plan_purges(d: Path):
    out = d / "outputs"
    if not out.exists():
        return
    if PURGE_RAW_JSON and (out / "Raw_Response.json").exists():
        actions.append(("delete_file", out / "Raw_Response.json"))
    if PURGE_PARSED_JSON and (out / "Parsed_Response.json").exists():
        actions.append(("delete_file", out / "Parsed_Response.json"))
    if PURGE_BATCH_ERRORS and (out / "Batch_Errors.json").exists():
        actions.append(("delete_file", out / "Batch_Errors.json"))

# 2) Zip older runs (into _archives)
def plan_zip(d: Path):
    z = ARCHIVE_DIR / f"{d.name}.zip"
    if not z.exists():
        actions.append(("zip_folder", (d, z)))

# 3) Delete runs beyond retention
to_prune = runs[KEEP_LAST_N_RUNS:] if KEEP_LAST_N_RUNS is not None else []
for d, ts in runs:
    if SKIP_CURRENT_RUN and current and d == current:
        continue
    # Purges
    plan_purges(d)
    # Zip plan
    if ZIP_OLDER_RUNS:
        plan_zip(d)

for d, ts in to_prune:
    actions.append(("delete_folder", d))

# ---------- Show plan ----------
print("\nPlanned actions:")
if not actions:
    print(" (none)")
else:
    for act, obj in actions:
        if act == "zip_folder":
            d, z = obj
            print(f" - ZIP {d.name}  →  {z.name}")
        else:
            print(f" - {act.upper()}: {obj}")

# ---------- Execute (unless DRY_RUN) ----------
if DRY_RUN:
    print("\nDRY_RUN=True — no changes applied. Set DRY_RUN=False to execute.")
else:
    for act, obj in actions:
        try:
            if act == "delete_file":
                Path(obj).unlink(missing_ok=True)
            elif act == "zip_folder":
                d, z = obj
                # create zip in ARCHIVE_DIR; shutil.make_archive adds .zip automatically
                base_name = z.with_suffix("")  # remove .zip for make_archive
                shutil.make_archive(str(base_name), 'zip', root_dir=d)
            elif act == "delete_folder":
                shutil.rmtree(obj, ignore_errors=True)
        except Exception as e:
            print("  ! Error:", act, obj, e)
    print("\n✅ Housekeeping complete.")

# ---------- Aggregate a master CSV across all runs (safe to do anytime) ----------
frames = []
for d, ts in runs:
    for name in ["Job_Classifications_Batch.csv", "Job_Classifications.csv"]:
        csvp = d / "outputs" / name
        meta = d / "RUN_METADATA.json"
        if csvp.exists():
            try:
                df_run = pd.read_csv(csvp)
                df_run["run_folder"]  = d.name
                df_run["source_file"] = name
                # enrich with metadata if available
                if meta.exists():
                    try:
                        m = json.loads(meta.read_text())
                        df_run["created_utc"] = m.get("created_utc")
                        df_run["model_used"]  = m.get("resolved_model_id") or m.get("model_used")
                    except Exception:
                        pass
                frames.append(df_run)
            except Exception as e:
                print(f"  ! Skipping {csvp.name} due to read error:", e)

if frames:
    master = pd.concat(frames, ignore_index=True)
    MASTER_DIR.mkdir(parents=True, exist_ok=True)
    master_out = MASTER_DIR / "All_Job_Classifications.csv"
    master.to_csv(master_out, index=False, encoding="utf-8")
    print(f"\n📚 Master CSV updated: {master_out} ({len(master)} rows; from {len(frames)} files)")
else:
    print("\n(No job classification CSVs found to aggregate.)")
